In [0]:
#import required module

from pyspark.sql.functions import *
from pyspark.sql import *
from pyspark.sql.types import *

In [0]:
#define storage account
Storage_name = "stytdev001"

#Configure Oauth authentication

spark.conf.set(
    f"fs.azure.account.auth.type.{Storage_name}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{Storage_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_yt_scope", key="app-yt-client-id"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_yt_scope", key="app-yt-client-secrets"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{Storage_name}.dfs.core.windows.net",
    "https://login.microsoftonline.com/65583477-d432-469c-b4af-ab046853916f/oauth2/v2.0/token",
)


In [0]:
#create widget for table name and storage path

dbutils.widgets.text('TABLE_NAME','')
dbutils.widgets.text('STORAGE_PATH','')

TABLE_NAME = dbutils.widgets.get('TABLE_NAME')
STORAGE_PATH = dbutils.widgets.get('STORAGE_PATH')

In [0]:
#define source and sink path

source_path = f"abfss://global@{Storage_name}.dfs.core.windows.net/bronze/"
sink_path = f"abfss://global@{Storage_name}.dfs.core.windows.net/silver/"


# STORAGE_FOLDER_NAME = TABLE_NAME + '_output/'

full_source_path = source_path + STORAGE_PATH + '/' + TABLE_NAME + '/'
full_sink_path = sink_path + STORAGE_PATH + '/' + TABLE_NAME

print(full_source_path)
print(full_sink_path)

abfss://global@stytdev001.dfs.core.windows.net/bronze/fare_rules_output/fare_rules/
abfss://global@stytdev001.dfs.core.windows.net/silver/fare_rules_output/fare_rules


In [0]:
#read and write data with requied information

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(full_source_path)\
     .withColumn("ACTION_TIMESTAMP", current_timestamp())\
     .withColumn("Created_by", lit("Tom"))\
     .withColumn("batch_id", date_format(current_timestamp(), "yyyyMMddHHmmss"))\
     .withColumn("_Source_file_Name", input_file_name())

df.write.format("csv").mode("overwrite").option("header", "true").save(full_sink_path)
